# Data Record Analysis and Visualization

**Author:** NUS PhotoMapper Team  
**Date:** 2026-09-24

## Objective
Analyze the core PhotoMapper record workbook and generate summary visualizations for time coverage and category distributions.

## Inputs
- `data/input/dataset_12663_with_disaster_type_refined.xlsx`

## Outputs
- `data/input/dataset_with_disaster_type.xlsx`
- Histogram figures and summary tables generated in notebook workflow.

In [ ]:
# If needed, uncomment and run the next line to install dependencies in your notebook kernel.
# %pip install pandas matplotlib seaborn openpyxl

import re
from pathlib import Path

import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

In [ ]:
# Use the refined dataset stored in the current project folder.
LOCAL_REFINED_FILE = Path("dataset_12663_with_disaster_type_refined.xlsx")
if not LOCAL_REFINED_FILE.exists():
    raise FileNotFoundError(f"Refined file not found in current folder: {LOCAL_REFINED_FILE}")

df = pd.read_excel(LOCAL_REFINED_FILE)
print(f"Loaded local refined file: {LOCAL_REFINED_FILE}")
print(f"Rows: {len(df):,}, Columns: {len(df.columns)}")
display(df.head(3))

In [ ]:
# Clean the refined disaster-type file and export a fresh dataset with standardized columns.
source_path = Path("dataset_12663_with_disaster_type_refined.xlsx")
output_path = Path("dataset_with_disaster_type.xlsx")

if source_path.exists():
    df_clean = pd.read_excel(source_path)
    df_clean.columns = [str(col).strip() for col in df_clean.columns]

    for col in df_clean.columns:
        if pd.api.types.is_object_dtype(df_clean[col]) or pd.api.types.is_string_dtype(df_clean[col]):
            df_clean[col] = df_clean[col].map(lambda v: str(v).strip() if pd.notna(v) else pd.NA)

    for col in ["CreationDate", "EditDate", "Photo_When", "Date_Time"]:
        if col in df_clean.columns:
            s = pd.to_datetime(df_clean[col].astype(str), errors="coerce", utc=True, format="mixed")
            df_clean[col] = s.dt.tz_localize(None)

    if "Incident" in df_clean.columns:
        df_clean["Incident"] = df_clean["Incident"].map(lambda v: str(v).strip() if pd.notna(v) else "")

    if "Disaster_Type" not in df_clean.columns and "Incident" in df_clean.columns:
        def classify_disaster_type(label: str) -> str:
            text = str(label).lower()
            if not text:
                return "Other / Unknown"

            cyclone_terms = [
                "hurricane", "typhoon", "cyclone", "tropical storm", "helene", "ida", "isaias",
                "sally", "ian", "debb", "milton", "idalia", "fiona", "henri", "sinlaku",
                "maria", "harvey", "irma", "laura", "katia", "katrina", "ophelia", "beryl",
                "zeta", "frances", "mitch", "nate", "cristobal", "francis", "michael", "sandy",
                "tropical"
            ]
            flood_terms = [
                "flood", "flooding", "inundation", "high water", "flash flood", "storm surge",
                "rainfall", "river flood", "coastal flooding", "tx flooding", "ca floods", "depth"
            ]
            tornado_terms = ["tornado", "twister", "funnel cloud"]
            earthquake_terms = ["earthquake", "seismic", "quake"]
            extreme_weather_terms = [
                "severe weather", "storm damage", "wind damage", "hail", "thunderstorm",
                "strong wind", "ice storm", "winter storm", "blizzard", "heat wave",
                "cold snap", "drought", "wind"
            ]
            wildfire_terms = ["wildfire", "wild fire", "forest fire", "brush fire", "fire", "burn", "burned area", "smoke"]

            if any(term in text for term in cyclone_terms):
                return "Tropical Cyclone"
            if any(term in text for term in flood_terms):
                return "Flood"
            if any(term in text for term in tornado_terms):
                return "Tornado"
            if any(term in text for term in earthquake_terms):
                return "Earthquake"
            if any(term in text for term in extreme_weather_terms):
                return "Extreme Weather"
            if any(term in text for term in wildfire_terms):
                return "Wildfire"
            return "Other / Unknown"

        df_clean["Disaster_Type"] = df_clean["Incident"].map(classify_disaster_type)

    if "Photo_ID" in df_clean.columns:
        df_clean = df_clean.drop_duplicates(subset=["Photo_ID"], keep="first")

    df_clean.to_excel(output_path, index=False)
    print(f"Saved cleaned dataset to: {output_path.resolve()}")
    display(df_clean.head())
else:
    print(f"Source file not found: {source_path}")

In [ ]:
# Global style: larger text, high contrast, clean layout (no background guidelines).
sns.set_theme(style="white")
FIG_SIZE = (14, 10)

plt.rcParams.update({
    "font.family": "sans-serif",
    "font.sans-serif": ["Avenir Next", "Avenir", "Helvetica Neue", "Gill Sans MT", "Trebuchet MS", "DejaVu Sans"],
    "figure.figsize": FIG_SIZE,
    "figure.dpi": 120,
    "axes.titlesize": 32,
    "axes.titleweight": "bold",
    "axes.labelsize": 26,
    "axes.labelweight": "bold",
    "xtick.labelsize": 20,
    "ytick.labelsize": 20,
    "legend.fontsize": 18,
    "axes.grid": False,
})

COLORS = {
    "year": "#1f77b4",
    "precise": "#ff7f0e",
    "lifelines": "#2ca02c",
    "disaster": "#d62728",
}

def standardize_text_series(series: pd.Series) -> pd.Series:
    out = series.astype("string").str.strip()
    out = out.replace({"": pd.NA, "nan": pd.NA, "None": pd.NA, "N/A": pd.NA})
    return out

def explode_multivalue(series: pd.Series) -> pd.Series:
    s = standardize_text_series(series).dropna()
    s = s.str.replace(r"\s*(;|\||/)\s*", ";", regex=True)
    return s.str.split(";").explode().str.strip().replace("", pd.NA).dropna()

def abbreviate_category_label(label: str) -> str:
    parts = re.split(r"[_\s/-]+", str(label).strip())
    parts = [p for p in parts if p]
    if not parts:
        return str(label)
    if len(parts) == 1:
        word = parts[0]
        return word[:4].upper() if len(word) > 4 else word.upper()
    return "".join(p[0].upper() for p in parts)

def make_unique_abbreviations(labels: pd.Index) -> list[str]:
    counts = {}
    out = []
    for label in labels.astype(str):
        base = abbreviate_category_label(label)
        seen = counts.get(base, 0) + 1
        counts[base] = seen
        out.append(base if seen == 1 else f"{base}_{seen}")
    return out

def add_bar_labels(ax: plt.Axes, fontsize: int = 22, padding: int = 4) -> None:
    for container in ax.containers:
        ax.bar_label(container, fmt="%.0f", fontsize=fontsize, padding=padding, weight="bold")

In [ ]:
# 1) Histogram: Year of record (dataset span in time) from CreationDate.
date_col = "CreationDate"
if date_col not in df.columns:
    raise ValueError(f"Column not found: {date_col}")

# The source uses mixed date formats (e.g. 2026-01-28 and 2026/02/02), so parse them
# explicitly as mixed to avoid dropping later 2026 values in the dataset.
date_parsed = pd.to_datetime(df[date_col].astype(str), errors="coerce", utc=True, format="mixed")
year_counts = date_parsed.dt.year.dropna().astype(int).value_counts().sort_index()



fig, ax = plt.subplots(figsize=FIG_SIZE)
bars = ax.bar(
    year_counts.index.astype(str),
    year_counts.values,
    color=COLORS["year"],
    edgecolor="black",
    linewidth=0.8,
)
add_bar_labels(ax, fontsize=22, padding=4)
# ax.set_title("Record Count by Year", pad=14, weight="bold")
# ax.set_xlabel("Year")
ax.set_ylabel("")
ax.tick_params(axis="x", rotation=45)
plt.tight_layout()
plt.savefig("figure_1_year_of_record.png", dpi=300, bbox_inches="tight")
plt.show()

print(f"Year span (CreationDate): {year_counts.index.min()} to {year_counts.index.max()}")

In [ ]:
# 2) Histogram: types of Precise_Location.
precise_col = "Precise_Location"
if precise_col not in df.columns:
    raise ValueError(f"Column not found: {precise_col}")

precise_counts = standardize_text_series(df[precise_col]).dropna().value_counts().head(20)

# Use fixed, readable abbreviations for known precise-location levels.
precise_abbr_map = {
    "Exact_Location": "Exact",
    "Street_Level": "Street",
    "City_Level": "City",
    "Neighborhood_Level": "Neighbor",
    "County_Level": "Country",
}
precise_labels_abbr = [precise_abbr_map.get(v, str(v)) for v in precise_counts.index.astype(str)]

precise_lookup = pd.DataFrame({
    "Abbr": precise_labels_abbr,
    "Original": precise_counts.index.astype(str),
    "Count": precise_counts.values,
})
print("Precise_Location abbreviation lookup:")
display(precise_lookup)

fig, ax = plt.subplots(figsize=FIG_SIZE)
sns.barplot(x=precise_counts.values, y=precise_labels_abbr, color=COLORS["precise"], ax=ax)
add_bar_labels(ax, fontsize=22, padding=4)
# Leave right-side room so the largest bar label stays inside the axes.
x_pad = precise_counts.max() * 0.08
ax.set_xlim(0, precise_counts.max() + x_pad)
#ax.set_title("Top Precise_Location Types", pad=14, weight="bold")
# ax.set_xlabel("Number of Records")
ax.set_ylabel("")
plt.tight_layout()
plt.savefig("figure_2_precise_location.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
# 3) Histogram: types of Community_Lifelines.
lifeline_col = "Community_Lifelines"
if lifeline_col not in df.columns:
    raise ValueError(f"Column not found: {lifeline_col}")

lifelines = explode_multivalue(df[lifeline_col])
lifeline_counts = lifelines.value_counts().head(20)
lifeline_labels_abbr = make_unique_abbreviations(lifeline_counts.index)

lifeline_lookup = pd.DataFrame({
    "Abbr": lifeline_labels_abbr,
    "Original": lifeline_counts.index.astype(str),
    "Count": lifeline_counts.values,
})
print("Community_Lifelines abbreviation lookup:")
display(lifeline_lookup)

fig, ax = plt.subplots(figsize=FIG_SIZE)
sns.barplot(x=lifeline_counts.values, y=lifeline_labels_abbr, color=COLORS["lifelines"], ax=ax)
add_bar_labels(ax, fontsize=22, padding=4)
# Leave right-side room so the largest bar label stays inside the axes.
x_pad = lifeline_counts.max() * 0.08
ax.set_xlim(0, lifeline_counts.max() + x_pad)
# ax.set_title("Top Community_Lifelines Types", pad=14, weight="bold")
# ax.set_xlabel("Number of Records")
ax.set_ylabel("")
plt.tight_layout()
plt.savefig("figure_3_community_lifelines.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
# 4) Histogram: disaster types.
# Prefer an explicit disaster-type column if a refined file already contains it.
disaster_type_fields = [
    "Disaster_Type",
    "disaster_type",
    "disaster_category",
    "DisasterCategory",
]
disaster_col = next((c for c in disaster_type_fields if c in df.columns), None)

# Parse and normalize noisy Incident strings into readable disaster labels.
def clean_disaster_label(v: str) -> str:
    if pd.isna(v):
        return pd.NA

    s = str(v).strip()
    if not s:
        return pd.NA

    s = re.sub(r"^CrowdsourcedPhoto[_\s]*", "", s, flags=re.IGNORECASE)
    s = s.replace(" ", "_")
    parts = [p for p in s.split("_") if p]

    if parts and re.fullmatch(r"\d{7,8}", parts[0]):
        parts = parts[1:]

    if parts and re.fullmatch(r"\d+|xxx|test", parts[-1], flags=re.IGNORECASE):
        parts = parts[:-1]

    if not parts:
        return pd.NA

    label = " ".join(parts)
    label = re.sub(r"\s+", " ", label).strip()
    return label if label else pd.NA

# Use a refined disaster type column when available, otherwise derive a category from the Incident labels.
def classify_disaster_type(label: str) -> str:
    if pd.isna(label):
        return "Other / Unknown"

    text = str(label).lower()
    if not text:
        return "Other / Unknown"

    cyclone_terms = [
        "hurricane", "typhoon", "cyclone", "tropical storm", "helene", "ida", "isaias",
        "sally", "ian", "debb", "milton", "idalia", "fiona", "henri", "sinlaku",
        "maria", "harvey", "irma", "laura", "katia", "katrina", "ophelia", "beryl",
        "zeta", "frances", "mitch", "nate", "cristobal", "francis", "michael", "sandy",
        "tropical"
    ]
    flood_terms = [
        "flood", "flooding", "inundation", "high water", "flash flood", "storm surge",
        "rainfall", "river flood", "coastal flooding", "tx flooding", "ca floods", "depth"
    ]
    tornado_terms = ["tornado", "twister", "funnel cloud"]
    earthquake_terms = ["earthquake", "seismic", "quake"]
    extreme_weather_terms = [
        "severe weather", "storm damage", "wind damage", "hail", "thunderstorm",
        "strong wind", "ice storm", "winter storm", "blizzard", "heat wave",
        "cold snap", "drought", "wind"
    ]
    wildfire_terms = ["wildfire", "wild fire", "forest fire", "brush fire", "fire", "burn", "burned area", "smoke"]

    if any(term in text for term in cyclone_terms):
        return "Tropical Cyclone"
    if any(term in text for term in flood_terms):
        return "Flood"
    if any(term in text for term in tornado_terms):
        return "Tornado"
    if any(term in text for term in earthquake_terms):
        return "Earthquake"
    if any(term in text for term in extreme_weather_terms):
        return "Extreme Weather"
    if any(term in text for term in wildfire_terms):
        return "Wildfire"
    return "Other / Unknown"

if disaster_col is not None and disaster_col not in ["Incident"]:
    disaster_values = df[disaster_col].map(classify_disaster_type)
else:
    df["Disaster_Type"] = df["Incident"].map(lambda v: classify_disaster_type(clean_disaster_label(v)))
    disaster_col = "Disaster_Type"
    disaster_values = df[disaster_col]

disaster_counts = disaster_values.value_counts().head(20)

fig, ax = plt.subplots(figsize=FIG_SIZE)
sns.barplot(x=disaster_counts.values, y=disaster_counts.index, color=COLORS["disaster"], ax=ax)
add_bar_labels(ax, fontsize=22, padding=4)
# ax.set_title(f"Top Disaster Types (from {disaster_col})", pad=14, weight="bold")
# ax.set_xlabel("Number of Records")
ax.set_ylabel("")
plt.tight_layout()
plt.savefig("figure_4_disaster_types.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
# Summary table: exact counts used in all four histograms.

def counts_to_table(series: pd.Series, histogram_name: str, category_name: str) -> pd.DataFrame:
    return pd.DataFrame({
        "Histogram": histogram_name,
        "Category": series.index.astype(str),
        "Count": series.values,
        "Category_Type": category_name,
    })

summary_table = pd.concat([
    counts_to_table(year_counts, "Year of Record", "Year"),
    counts_to_table(precise_counts, "Precise_Location Types", "Precise_Location"),
    counts_to_table(lifeline_counts, "Community_Lifelines Types", "Community_Lifelines"),
    counts_to_table(disaster_counts, f"Disaster Types (from {disaster_col})", "Disaster_Type"),
], ignore_index=True)

# Keep rows grouped by histogram and sorted by count within each group.
summary_table["Histogram"] = pd.Categorical(
    summary_table["Histogram"],
    categories=[
        "Year of Record",
        "Precise_Location Types",
        "Community_Lifelines Types",
        f"Disaster Types (from {disaster_col})",
    ],
    ordered=True,
)
summary_table = summary_table.sort_values(["Histogram", "Count"], ascending=[True, False]).reset_index(drop=True)

csv_path = Path("histogram_summary_table.csv")
summary_table.to_csv(csv_path, index=False)

print(f"Total rows in summary table: {len(summary_table):,}")
print(f"CSV saved to: {csv_path.resolve()}")

display(
    summary_table.style
    .set_caption("Exact Counts for All Histograms")
    .format({"Count": "{:,.0f}"})
    .set_properties(**{"font-size": "16px"})
    .set_table_styles([
        {"selector": "caption", "props": [("font-size", "20px"), ("font-weight", "bold"), ("text-align", "left")]},
        {"selector": "th", "props": [("font-size", "16px"), ("font-weight", "bold")]},
    ])
)

summary_table.head()